# 01 - Data cleaning and quality report

This step creates cleaned copies of all supplied CSVs and a machine-readable audit. Raw inputs stay unchanged. The cleaning implementation is in `src/data_cleaning.py`; this notebook runs it and provides review tables.

## Decisions that prevent silent data loss
- Parse and standardize dates; use `Date` as authoritative and recompute `Year`/`Month`. Keep raw values and flag periods outside the stated history/forecast window.
- For repeated `Record_ID`, keep the latest `Load_Timestamp`; retain source row numbers and report conflicting duplicate groups.
- Standardize known channel/region aliases. Fill missing SKU family, business unit, and description from `sku_master.csv`; correct conflicting business unit from that reference while preserving the original value.
- Treat blank and `-` units as missing, never zero. Keep actual zero sales as zero.
- Correct zero or clear powers-of-ten price errors only when same-SKU/month peers provide evidence. Flag other price deviations for event/source review.
- Recompute revenue from units and cleaned price when both are available; retain `Revenue_USD_Raw` for comparison.
- Keep unresolved duplicate forecast-history keys and conflicting macro months visible; do not aggregate without evidence.

Install the project dependencies from the project root with `uv pip install --python .venv\\Scripts\\python.exe -r requirements.txt` if the environment has not already been set up.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "student_data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/student_data from the notebook working directory.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw inputs: {PROJECT_ROOT / 'data' / 'student_data'}")
print(f"Cleaned outputs: {PROJECT_ROOT / 'outputs' / 'cleaned'}")
print(f"Quality reports: {PROJECT_ROOT / 'outputs' / 'data_quality'}")

In [ ]:
from src.data_cleaning import run_data_cleaning

audit = run_data_cleaning(PROJECT_ROOT)
print("Cleaning completed.")
print(f"Cleaned files: {len(audit['file_metrics'])}")
print(f"Quality findings: {len(audit['issues'])}")
print(f"Documented corrections: {len(audit['corrections'])}")

In [ ]:
import pandas as pd
from IPython.display import display

file_metrics = pd.DataFrame.from_dict(audit["file_metrics"], orient="index").reset_index(names="file")
display(file_metrics)

issues = pd.read_csv(PROJECT_ROOT / "outputs" / "data_quality" / "data_quality_issues.csv")
corrections = pd.read_csv(PROJECT_ROOT / "outputs" / "data_quality" / "data_corrections.csv")

print("\nIssues requiring review")
display(issues.sort_values(["severity", "affected_rows"], ascending=[True, False]))

print("\nApplied corrections")
display(corrections.sort_values("affected_rows", ascending=False))

import numpy as np
clean_train = pd.read_csv(PROJECT_ROOT / "outputs" / "cleaned" / "train_cleaned.csv")
clean_test = pd.read_csv(PROJECT_ROOT / "outputs" / "cleaned" / "test_cleaned.csv")
clean_skus = pd.read_csv(PROJECT_ROOT / "outputs" / "cleaned" / "sku_master_cleaned.csv")
valid_train_keys = clean_train["Composite_Key_Valid"].fillna(False).astype(bool)
assert len(clean_train) == audit["file_metrics"]["train.csv"]["output_rows"]
assert len(clean_test) == 58_219
assert not clean_train.loc[valid_train_keys].duplicated(["Date", "SKU_ID", "Region"]).any()
assert not clean_test.duplicated(["Date", "SKU_ID", "Region"]).any()
assert clean_train.loc[
    clean_train["Units_Sold_Raw"].astype(str).str.strip().isin(["", "-"]), "Units_Sold"
].isna().all()
reconciled = clean_train[["Units_Sold", "Unit_Price_USD", "Revenue_USD"]].dropna()
assert np.allclose(
    reconciled["Units_Sold"] * reconciled["Unit_Price_USD"],
    reconciled["Revenue_USD"], atol=0.01, rtol=0,
)
assert clean_train["SKU_ID"].isin(clean_skus["SKU_ID"]).all()
print("\nValidation passed: expected test coverage, unique complete keys, preserved missing targets, and reconciled revenue.")

## Outputs and handoff
- Cleaned datasets are in `outputs/cleaned/`.
- `outputs/data_quality/data_quality_issues.csv` lists issue type, affected rows, severity, handling decision, and examples.
- `outputs/data_quality/data_corrections.csv` lists transformations and affected-row counts.
- `outputs/data_quality/data_quality_summary.json` records per-file row counts, key cleaning metrics, decisions, and assumptions.

Before continuing, review unresolved price deviations, the out-of-window training dates, and any remaining missing targets. Do not treat unresolved missing units as zero demand.